<a href="https://colab.research.google.com/github/wlgns222/ROKA/blob/main/ai-study/deep-learning-from-scratch-vol1/Ch6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ch6. Training Techniques

## 6.1 Optimization

### 6.1.2 확률적 경사 하강법 (SGD)

**SGD 란**

기울어진 방향으로 일정 거리만큼을 가겠다는 방법

In [ ]:
class SGD :
  def __init__(self, lr = 0.01) :
    self.lr = lr
  def update(self, params, grads):
    for key in params.keys():
      params[key] -= self.lr * grads[key]

- 인수 lr : learning rate 학습률
- 메서드 update(params, grads)
  - 인수 params, grads : params['W1'], grads['W1'] 등과 같이 매개변수와 기울기를 저장

SGD 클래스를 사용하면 신경망 매개변수 진행을 다음과 같이 수행 가능하다.

In [ ]:
network = TwoLayerNet(...)
optimizer = SGD()

for i in range(10000) :
  ...
  x_batch, t_batch = get_mini_batch(...)
  grads = network.gradient(x_batch, t_batch)
  params = network.params

  optimizer.update(params, grads)
  ...

**optimizer : 최적화를 행하는 자**

최적화를 담당하는 클래스를 분리해 구현하면 기능을 모듈화 하기 좋다.

Momentum 이라는 최적화 기법 역시 update(params, grads) 라는 메서드를 갖도록 구현하고, optimizer = SGD() 문장을 optimizer = Momentum()으로 변경하면 간단하게 바꿀 수 있다.

### 6.1.3 SGD의 단점

SGD는 단순하고 구현도 쉬우나, 문제에 따라서 비효율적일 때가 있다.

<br>


**SGD는 비등방성(anisotropy) 함수에서 비효율적이다.**

비등방성이란 방향에 따라 물리적 성질이 바뀌는 것이다. 즉, 기울기가 가르키는 지점이 하나(최솟값)가 아니라 여러가지일 때 기울기가 가르키는 방향으로 이동하는 SGD는 비효율적인 움직임을 보인다.


SGD의 이런 단점을 개선해주는 optimizer
- Momentum
- AdaGrad
- Adam

### 6.1.4 모멘텀 (Momentum)

**모멘텀 (Momentum) 이란**

'운동량'을 뜻하는 단어로 물리와 관계가 있다. 모멘텀 기법은 수식으로 다음과 같이 쓸 수 있다.

1. $v \leftarrow \alpha v - \eta \frac{\partial L}{\partial W}$

2. $W \leftarrow W + v$

SGD 에서와 마찬가지로 $W$ 는 갱신할 매개변수, $\frac{\partial L}{\partial W}$ 은 $W$ 에대한 손실함수의 기울기, $\eta$ 는 학습률이다.

$v$ 라는 변수가 새로 나오는데, 이는 물리에서의 속도에 해당한다.

$\alpha$ 는 마찰력같은 존재로 보통 0.9로 설정하여 과거의 속도를 어느정도 유지하며 가속도를 붙게한다.

**모멘텀은 공이 기울기를 따라 구르듯 움직인다**



In [ ]:
import numpy as np

class Momentum :
  def __init__ (self, lr = 0.01, momentum = 0.9):
    self.lr = lr
    self.momentum = momentum
    self.v = None
  def update(self, params, grads) :
    #Initialize v buffer on the first run
    if self.v is None :
      self.v = {}
      for key, val in params.items():
        self.v[key] = np.zero_like(val)
    for key in params.keys():
      # v = alpha * v - lr * grads
      self.v[key] = self.momentum * self.v[key] - self.lr * grads[key]
      #W = W + v
      params[key] += self.v[key]

**모멘텀의 장점**
1. 지그재그로 요동치는 구간에서 좌우 움직임은 서로 상쇄한다.
2. 목표를 향한 전진방향에 가속도를 붙임으로서 더 빠르게 손실함수의 바닥에 도달할 수 있다.

### 6.1.5 아다그라드 (AdaGrad)

신경망 학습에서 학습률 값이 매우 중요하다. 값이 너무 작으면 학습 시간이 길어지며, 너무 크면 발산하여 학습이 제대로 이뤄지지 않는다.

<br>

**아다그라드 (AdaGrad) 의 핵심 : '학습률 감소'**

아다그라드는 '학습률 감소'를 매개변수마다 다르게 적용하여, 각각의 매개변수에 맞춤형 값을 제공한다. 많이 움직인 매개변수는 학습률을 대폭 줄여서 세밀하게 조정하고, 적게 움직인 매개변수는 학습률을 유지하여 더 공부할 기회를 준다. 이를 수식으로 나타내면 다음과 같다.

1. $h \leftarrow h + (\frac{\partial L}{\partial W})^2$

2. $W \leftarrow W - \eta \frac{1}{\sqrt{h}} \frac{\partial L}{\partial W}$

마찬가지로 $W$ 는 갱신할 매개변수, $\frac{\partial L}{\partial W}$ 은 $W$ 에대한 손실함수의 기울기, $\eta$ 는 학습률이다.

$h$ 는 기존 기울기 값을 제곱하여 계속 더해준다. 그리고 매개변수를 갱신할 때 $\frac{1}{\sqrt{h}}$ 를 곱하여 학습률을 조정한다.


In [ ]:
class AdaGrad :
  def __init__ (self, lr = 0.01) :
    self.lr = lr
    self.h = None
  def update (self, params, grads) :
    if self.h is None :
      self.h = {}
      for key, val in params.items() :
        self.h[key] = np.zeros_like(val)
    for key in params.keys() :
      #Square of current gradient
      self.h[key] += grads[key] * grads[key]
      #Update params : divide by sqrt(h)
      #Add 1e-7 to avoid division by zero
      params[key] -= self.lr * grads[key] / (np.sqrt(self.h[key]) + 1e-7)

**Note**

AdaGrad 는 과거의 기울기를 제곱하여 계속 더해감으로 $h$ 가 무한히 커지고 학습률이 0이 되어 '학습정지' 상태에 빠질 수 있다는 단점이 있다.

이를 보완하기 위해 **RMSProps** 라는 방법이 있다. 이는 먼 과거의 기울기는 잊고 새로운 기울기 정보를 크게 반영하는 기법이다.

### 6.1.6 아담 (Adam)

**아담 (Adam) 이란**

*Momentum* 의 장점과 *AdaGrad* 의 장점을 결합한 기법이다.

2015년에 제안된 방법이며, 현대 딥러닝 프로젝트에서 가장 많이 쓰이는 범용성이 뛰어난 엔진이다.

아담의 특징으로 **편향 보정** (Bias Correction) 이 있다.

학습 초기에는 누적된 데이터가 없어 속도나 학습률 조절이 부정확할 수 있는데, 이를 수학적으로 보정하여 첫 번째 스텝부터 엔진이 최고 출력을 낼 수 있도록 돕는 방법이다.

**Note**

Adam 은 3개의 하이퍼파라미터를 사용한다.
1. 학습률 - $α$
2. 일차 모멘텀용 계수 - $\beta_1=0.9$
3. 이차 모멘텀용 계수 - $\beta_2=0.999$

## 6.2 Weight Initialization

가중치의 초깃값을 무엇으로 설정하여야 할까?

### 6.2.1 초깃값을 0으로 하면?

**초깃값을 0으로 설정하면 안된다.**

가중치 값이 모두 0인 경우, 모든 뉴련은 똑같은 계산을 하고 똑같이 갱신된다. 따라서 가중치들은 같은 초깃값에서 시작하고 갱신을 거치더라도 여전히 같은 값을 유지한다.

이는 가중치를 여러개 갖는 의미를 사라지게 하는 효과를 낳는다.

이러한 문제를 **대칭성** 문제라고 하며, 이를 깨기위해 가중치에 **무작위성**을 부여해야한다.

### 6.2.2 은닉층의 활성화 값 분포

**가중치의 분포에 따른 활성화 값 분포**

Case.1 표준편차가 1인 정규 분포
- 각 층의 활성화 값들이 0과 1에 치우쳐서 분포된다.
- 데이터가 0과 1에 치우쳐 분포하게 되면 역전파의 기울기 값이 점점 작아지다가 사라진다. 해당 현상을 **기울기 소실**이라고 한다.

Case.2  표준편차가 0.01인 정규분포
- 활성화 값들이 0.5 부근에 집중되어 나타난다.
- **표현력 제한**이라는 문제가 발생한다.

### 6.2.2 Xavier 초기값

**Xavier 초기값이란**

앞 계층의 노드 $n$ 에 대하여, 표준편차가 $\frac{1}{\sqrt{n}}$ 인 분포를 가중치 초기값에 사용하는 것

앞 층에 노드가 많을수록 대상 노드의 초기값을 설정하는 가중치가 좁게 퍼진다.

tanh 함수, sigmoid 함수와 같이 대칭적인 함수에 사용하기 적절하다.

### 6.2.3 He 초기값

ReLU 함수를 이용할 때에는 그에 특화된 초깃값을 이용하라고 권장한다. 해당 초깃값이 **He 초깃값**이다.

**He 초기값이란**

앞 계층의 노드 $n$ 에 대하여, 표준편차가 $\sqrt{\frac{2}{n}}$ 인 분포를 가중치 초기값에 사용하는 것

ReLU 함수는 음의 영역이 0이라 더 넓게 분포시키기 위해 $\sqrt{2}$ 배 넓은 계수가 필요하다.

현대 딥러닝 주력인 ReLU 를 사용할 때 필수적인 초기값이다.

## 6.3 Batch Normalization

### 6.3.1 배지 정규화 알고리즘

**배치 정규화의 기본 아이디어**

각 층의 활성화 값 분포가 적절하면 학습이 원활히 수행된다.

배치 정규화는 각 층의 활성화를 적당히 퍼뜨리도록 강제한다.

**배치 정규화의 장점**

배치 정규화는 2015년도에 제안된 방법이며, 많은 연구자와 기술자가 즐겨 사용한다. 실제 머신러닝 콘테스트의 결과상으로도 배치 정규화를 사용하여 뛰어난 결과를 달성한 예가 많다.

배치 정규화의 장점은 다음과 같다.
1. 학습을 빨리 진행할 수 있다.
2. 초깃값에 크게 의존하지 않는다.
3. 과대적합을 억제한다.

**배치 정규화 계층**

- 배치 정규화는 학습 시 미니배치 단위로 정규화 한다. 데이터 분포가 평균이 0, 분산이 1이 되도록 강제한다.
- 주로 행렬 곱(Affine) 직후 ReLU 함수 사이에 위치한다.

<br>

**배치 정규화 수식**

미니배치 $B = \{x_1, x_2, \dots, x_m\}$ 에 대해 다음 절차를 수행한다.

미니배치 $B$에 대해 평균  $\mu_B$ 과 분산 $\sigma_B^2$ 을 구한다.
1. 미니배치 평균 : $\mu_B \leftarrow \frac{1}{m} \sum_{i=1}^m x_i$
2. 미니배치 분산 : $\sigma_B^2 \leftarrow \frac{1}{m} \sum_{i=1}^m (x_i - \mu_B)^2$

입력 데이터를 평균이 0, 분산이 1이 되도록 정규화한다.

3. 정규화 : $\hat{x}_i \leftarrow \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$

정규화된 데이터에 대해여 확대와 이동 변환(Scale and Shift)을 수행한다.

4. Scale and Shift : $y_i \leftarrow \gamma \hat{x}_i + \beta$

해당 식에서 $\gamma$ 가 확대를, $\beta$ 가 이동을 담당한다. 처음에는 각각 1과 0으로 시작하고, 학습하면서 적절한 값으로 조절한다.

### 6.3.2 배치 정규화의 효과

1. 빠른 학습
    - 학습률을 높게 잡아도 폭주하지 않아 학습 속도가 비약적으로 증가한다.
2. 가중치 초기값에 의존하지 않음
    - 가중치 초기값을 잘못 설정하더라도 알아서 교정해준다.
3. 과대 적합을 억제한다.
    - 드롭아웃의 역할을 작게나마 수행해주며 오버피팅을 억제하는 효과가 있다.

## 6.4 Regularization

머신러닝에는 **과대적합**이 문제가 되는 일이 많다.

과대적합이란 신경망이 훈련 데이터에만 지나치게 적응되어 그 외의 데이터에는 제대로 대응하지 못하는 상태를 말한다.

### 6.4.1 Overfitting

과대적합은 주로 다음 두 경우에 일어난다.
- 매개변수가 많고 표현력이 높은 모델
- 훈련 데이터가 적음

### 6.4.2 Weight Decay

과대적합 억제용으로 많이 이용해온 방법 중 **가중치 감소**가 있다.

**가중치 감소**는 학습 과정에서 큰 가중치에 대해서는 그에 상응하는 큰 페널티를 부과하여 과대적합을 억제하는 방법이다.

**가중치 감소의 원리**

신경망의 목적은 손실 함수의 값을 줄이는 것이다. 이때 가중치의 제곱 노름(L2 노름)을 손실 함수에 더해주면 가중치가 커지는 현상을 억제할 수 있다.

가중치를 $W$ 라고 했을 때, L2 노름에 따른 가중치 감소는 $\frac{1}{2}\lambda W^2$ 이 되고, 해당 값을 손실 함수에 더해준다.

여기에서 $\lambda$ 는 정규화의 세기를 조절하는 하이퍼파라미터이다. 해당 값의 크기가 클수록 큰 가중치에 대한 페널티가 커진다.

### 6.4.3 Dropout

가중치 감소는 간단하게 구현할 수 있으며, 과대적합을 억제할 수 있다. 그러나 모델이 복잡해지면 가중치 감소만으로는 대응하기 힘들다.

이럴 때 흔히 **드롭아웃** 이라는 기법을 사용한다.

**드롭아웃의 원리**

드롭아웃은 뉴런을 임의로 삭제하며 학습하는 기법이다.

훈련 때 은닉층의 뉴런을 무작위로 골라 삭제하며, 삭제된 뉴런은 신호를 전달하지 않게 된다.

훈련 때는 데이터를 흘릴 때마다 삭제할 뉴런을 무작위로 선택했고, 시험 때는 모든 뉴런에 신호를 전달한다.

In [ ]:
class Dropout:
  def __init__ (self, dropout_ratio=0.5) :
    self.dropout_ratio = dropout_ratio
    self.mask = None
  def forward(self, x, train_flg = Ture) :
    if train_flg :
      self.mask = np.random.rand(*x.shape) > self.dropout_ratio
      return x * self.mask
    else :
      return x * (1.0 - sefl.drop_ratio)
  def backward (self, dout) :
    return dout * self.mask

**앙상블 학습**

앙상블 학습이란 개별적으로 학습시킨 여러 모델의 출력을 평균 내어 추론하는 방식이다.


드롭아웃은 훈련 시 뉴런을 무작위로 꺼버려 마치 여러 개의 작은 신경망을 동시에 훈련시켜 평균을 내는 '앙상블 효과'를 준다.

## 6.5 Hyperparameter Optimization

### 6.5.1 검증 데이터

하이퍼파라미터의 성능을 평가할 때 **검증 데이터** 가 필요하다.

하이터파라미터 성능 평가에 시험 데이터를 사용하게 된다면 값들이 시험 데이터에 과대적합이 되어 범용 성능이 떨어지게 된다.

따라서 하이퍼파라미터를 조정할 때는 하이터파라미터 전용 확인 데이터 **검증 데이터**가 필요하다.

- 훈련 데이터 : 매개변수 학습
- 검증 데이터 : 하이퍼파라미터 성능 평가
- 시험 데이터 : 신경망의 범용 성능 평가

### 6.5.2 하이퍼파라미터 최적화

하이퍼파라미터를 최적화 할 때의 핵심은 하이퍼파라미터의 '최적값'이 존재하는 범위를 조금씩 줄여나가는 것이다.



**0단계. 범위 설정**
  
  하이퍼파라미터 값의 범위를 설정한다.
  
  0.001에서 1000 사이 같이 '10의 거듭제곱' 단위로 범위를 지정한다. 이를 **로그 스케일**이라고 한다.

**1단계. 무작위 추출**
  
  설정된 범위에서 하이퍼파라미터의 값을 무작위로 추출한다.
  
  그리드 서치 같은 규칙적 탐색보다는 무작위로 샘플링해 탐색하는 편이 좋은 결과를 낸다.

**2단계. 하이퍼파라미터 평가**

  1단계에서 샘플링한 하이터파라미터 값을 사용하여 학습하고, 검증 데이터로 정확도를 평가한다.
  
  하이퍼파라미터를 최적화 할 때 딥러닝 학습에 매우 오랜 시간이 걸리기 때문에 학습을 위한 에포크를 작게하여 1회 평가에 걸리는 시간을 단축하는 편이 효과적이다.

**3단게 : 반복**

  1단계와 2단계를 특정 횟수 반복하여, 그 정확도의 결과를 보고 하이퍼파라미터의 범위를 좁힌다.
  
  어느 정도 좁아지면 그 압축한 범위에서 값을 골라낸다.

## 6.6 정리

이번 장에서는 신경망 학습에 중요한 기술 몇 가지를 학습하였다.

이번 장에서 배운 내용
- 매개변수 갱신 방법에는 확률적 경사 하강법 외에도 모멘텀, AdaGrad, Adam 이 있다.
- 가중치 초깃값을 정하는 방법은 올바른 학습을 하는 데 매우 중요하다.
- 가중치의 초깃값으로는 'Xavier 초깃값'과 'He 초깃값'이 효과적이다.
- 배치 정규화를 이용하면 학습을 빠르게 진행할 수 있으며, 초깃값의 영향을 덜 받는다.
- 과대적합을 억제하는 정규화 기술로는 가중치 감소와 드롭아웃이 있다.
- 하이퍼파라미터 값 탐색은 최적 값이 존재할 법한 범위를 점차 좁히면서 하는 것이 효과적이다.